In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file=f"{landing_folder_path}/constructors.json"
table_name=f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Define schema for constructors JSON file with StructType and StructField
constructors_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

# Read constructors JSON file from volume with defined schema
constructors_df = (spark.read
    .schema(constructors_schema)
    .option("mode", "FAILFAST")
    .json(source_file)
)

#display(constructors_df)

In [0]:
# Add ingestion timestamp and source file path metadata using helper function
constructors_with_metadata_df = add_ingestion_metadata(constructors_df)

# display(constructors_with_metadata_df)

In [0]:
# Write constructors data to bronze layer table
(constructors_with_metadata_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"✅ Constructors data successfully written to {table_name} table")

In [0]:
%sql
-- SELECT 
--   constructorId,
--   name,
--   nationality,  
--   url,
--   ingestion_timestamp,
--   source_file_path
-- FROM formula1.bronze.constructors
-- LIMIT 10